In [1]:
import json

with open('example.json') as f:
    d = json.load(f)

In [2]:
import sys
import os
from pathlib import Path

root_dir = Path(os.getcwd()).parent / 'app'
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

In [3]:
from db.db import get_session, init_db

init_db()

#with get_session() as session:

2026-05-14 21:48:06,892 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-05-14 21:48:06,892 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-14 21:48:06,918 INFO sqlalchemy.engine.Engine select current_schema()
2026-05-14 21:48:06,919 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-14 21:48:06,921 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-05-14 21:48:06,922 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-14 21:48:06,923 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-14 21:48:06,924 INFO sqlalchemy.engine.Engine COMMIT


In [4]:
class AutoDBDict:
    def __init__(self) -> None:
        self.stmts = {}
    
    def set_stmt(self, object_type, stmt):
        self.stmts[object_type] = stmt

    def add_to_db(self, object, session):
        db_object = session.exec(self.stmts[type(object)](object)).first()
        if not db_object:
            db_object = object
            session.add(db_object)
            session.flush()
        return db_object

to_db = AutoDBDict()

In [ ]:
from models.product import Product
from models.attribute import Attribute
from models.attribute_name import AttributeName
from models.property_name import PropertyName
from models.property import Property
from models.variant import Variant
from sqlmodel import select

to_db.set_stmt(Product, lambda obj: select(Product).where(Product.name == obj.name))
to_db.set_stmt(AttributeName, lambda obj: select(AttributeName).where(AttributeName.name == obj.name))
to_db.set_stmt(Attribute, lambda obj: select(Attribute).where(Attribute.attribute_name_id == obj.attribute_name_id, Attribute.value == obj.value))
to_db.set_stmt(PropertyName, lambda obj: select(PropertyName).where(PropertyName.name == obj.name))
to_db.set_stmt(Property, lambda obj: select(Property).where(Property.property_name_id == obj.property_name_id, Property.value == obj.value, Property.attribute_id == obj.attribute_id))
to_db.set_stmt(Variant, lambda obj: select(Variant).where(Variant.product == obj.product))

id_to_att_prop = {}

with get_session() as session:
    product_db = to_db.add_to_db(Product(name=d['data']['result']['GLOBAL_DATA']['globalData']['subject']), session)
    
    for att in d['data']['result']['PRODUCT_PROP_PC']['showedProps']:
        an = to_db.add_to_db(AttributeName(name=att['attrName']), session)
        to_db.add_to_db(Attribute(attribute_name_id=an.id, value=att['attrValue']), session)

    for prop in d['data']['result']['SKU']['skuProperties']:
        an = to_db.add_to_db(AttributeName(name=prop['skuPropertyName']), session)
        for val in prop['skuPropertyValues']:
            a = to_db.add_to_db(Attribute(attribute_name_id=an.id, value=val['propertyValueName']), session)
            id_to_att_prop[f'{prop['skuPropertyId']}:{val['propertyValueIdLong']}'] = (a, [])
            if "propertySizeChartInfo" in val:
                for p in val["propertySizeChartInfo"]:
                    pn = to_db.add_to_db(PropertyName(name=p['name']), session)
                    id_to_att_prop[f'{prop['skuPropertyId']}:{val['propertyValueIdLong']}'][1].append((pn.id, p['value']))
    
    variants = {}

    for key, val in d['data']['result']['PRICE']['skuIdStrPriceInfoMap'].items():
        variants[key] = {"price": float(val['salePriceString'][:-2].replace(" ", "").replace(',', '.'))}

    for path in d['data']['result']['SKU']['skuPaths']:
        variants[path['skuIdStr']]['attributes'] = []
        att_combination = path['path'].split(";")
        for seq in att_combination:
            variants[path['skuIdStr']]['attributes'].append(id_to_att_prop[seq][0])

        var = to_db.add_to_db(Variant(product_id=product_db.id, price=int(variants[path['skuIdStr']]['price']), stock=path['skuStock'], attributes=variants[path['skuIdStr']]['attributes']), session)

        for seq in att_combination:
            if len(id_to_att_prop[seq][1]) != 0:
                for prop in id_to_att_prop[seq][1]:
                    to_db.add_to_db(Property(property_name_id=prop[0], attribute_id=id_to_att_prop[seq][0].id, variant_id=var.id, value=prop[1]), session)
                
        print(f"Variant: {var}")

    session.commit()

2026-05-14 23:31:31,440 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-14 23:31:31,442 INFO sqlalchemy.engine.Engine SELECT product.id, product.name, product.description 
FROM product 
WHERE product.name = %(name_1)s
2026-05-14 23:31:31,443 INFO sqlalchemy.engine.Engine [cached since 2087s ago] {'name_1': "New Fashion Men's Women's Hoodies Spring Autumn Winter Casual Hoodies Sweatshirts Men Tops Solid Color Hoodie Sweatshirt Male"}
2026-05-14 23:31:31,445 INFO sqlalchemy.engine.Engine SELECT attributename.id, attributename.name 
FROM attributename 
WHERE attributename.name = %(name_1)s
2026-05-14 23:31:31,447 INFO sqlalchemy.engine.Engine [cached since 2086s ago] {'name_1': 'whether full opening'}
2026-05-14 23:31:31,449 INFO sqlalchemy.engine.Engine SELECT attribute.id, attribute.attribute_name_id, attribute.value 
FROM attribute 
WHERE attribute.attribute_name_id = %(attribute_name_id_1)s AND attribute.value = %(value_1)s
2026-05-14 23:31:31,450 INFO sqlalchemy.engine.Engine 

s:\Python\Python313\Lib\site-packages\sqlmodel\orm\session.py:75: SAWarning: Object of type <Variant> not in session, add operation along 'Attribute.variants' won't proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  results = super().execute(


2026-05-14 23:31:31,986 INFO sqlalchemy.engine.Engine [cached since 1021s ago] {'property_name_id_1': 5, 'value_1': '106cm', 'attribute_id_1': 306}
2026-05-14 23:31:31,988 INFO sqlalchemy.engine.Engine SELECT property.id, property.property_name_id, property.variant_id, property.attribute_id, property.value 
FROM property 
WHERE property.property_name_id = %(property_name_id_1)s AND property.value = %(value_1)s AND property.attribute_id = %(attribute_id_1)s
2026-05-14 23:31:31,989 INFO sqlalchemy.engine.Engine [cached since 1021s ago] {'property_name_id_1': 6, 'value_1': '68cm', 'attribute_id_1': 306}
2026-05-14 23:31:31,990 INFO sqlalchemy.engine.Engine SELECT property.id, property.property_name_id, property.variant_id, property.attribute_id, property.value 
FROM property 
WHERE property.property_name_id = %(property_name_id_1)s AND property.value = %(value_1)s AND property.attribute_id = %(attribute_id_1)s
2026-05-14 23:31:31,991 INFO sqlalchemy.engine.Engine [cached since 1021s ago] 

Product()